# TRACE-RAG: full BEIR NQ end-to-end run on Colab NVIDIA T4

This notebook runs the full **2,681,468-passage BEIR Natural Questions corpus** through TRACE-RAG, uses a real locally served **Ollama Qwen2.5 3B** generator, and exercises the Person A retrieval/generation path, Person B trust/verification/remediation path, and Person C controlled attacks/evaluation code. It also runs a one-query diagnostic through every implemented Person C baseline policy. The complete repository test suite runs first.

## Important scope and data-integrity notes

- The corpus, queries, and qrels are authentic BEIR NQ data. The full passage corpus is indexed; 500 genuine qrels-backed queries are sampled deterministically for the stream, so this is **not** evaluation over every NQ query.
- The notebook requires an NVIDIA T4 CUDA runtime. Contriever embeddings run on CUDA in float16, and NLI verification explicitly requests CUDA; batch size is selected from reported VRAM. FAISS IVF-PQ, SQLite, parsing, and most application logic remain on CPU. Ollama serves a real local model and auto-selects available GPU/CPU resources; inspect `ollama ps` output rather than assuming it used the GPU.
- BEIR NQ has no contributor identities, original ingestion timestamps, or real poisoning incidents. To avoid inventing a contributor roster or fake historical ages, the run derives page-level source IDs from each authentic BEIR Wikipedia title and records wall-clock import times. Its attacker identity and poison payloads are controlled test inputs: each poison is a generated corruption of a real qrels-relevant NQ passage. These are not authentic incident records or organic attacks. A literal zero-generated-content study requires real source metadata and real incident documents, which this dataset does not supply.
- The trust-stream output and one-query baseline output are integration diagnostics, **not** a complete attack × baseline × seed benchmark or publication-ready result. The baseline diagnostic uses authentic retrieved NQ documents, but only one query; the complete Person C matrix is still a separate research harness gap. Unit tests use small test fixtures as normal software tests; they are not benchmark data.
- Full-corpus ingestion/indexing is large and may take hours and substantial disk/RAM. Keep enough free Colab storage for Parquet, SQLite, checkpoints, model weights, and outputs. This notebook is experimental and has not been live-validated on a T4 in this development environment.


## Before running

1. In Colab select **Runtime → Change runtime type → T4 GPU**, then restart the runtime. The notebook fails fast unless CUDA is visible and the detected GPU name includes T4.
2. Run every cell from the top in a fresh runtime. Do not select a TPU runtime for this notebook.
3. The install cell adds project, FAISS, Parquet, plotting, testing, and Transformers dependencies without requesting the `models` extra or replacing Colab's CUDA-enabled PyTorch.
4. Allow internet access to Hugging Face and Ollama. The run downloads the full NQ parquet, Contriever/NLI weights, the Qwen2.5 3B Ollama model, and a small DistilGPT-2 model used only by the perplexity-baseline diagnostic.
5. Budget for several GB of model and dataset downloads plus potentially much larger indexed SQLite/FAISS artifacts. The final ZIP excludes the downloaded NQ corpus.

The notebook clones the fixed Arena branch below. It never embeds GitHub credentials.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import time
import gc
import datetime
import csv
import zipfile
import platform
import importlib.metadata

REPO_URL = "https://github.com/athishss/-Adaptive-Trust-Based-Framework-for-Detecting-and-Preventing-Knowledge-Base-Poisoning-in-RAG-Systems.git"
BRANCH = "arena/aac1ce08-adaptive-trust-based-framework"
REPO = Path("/content/trace-rag-t4")

if not (REPO / ".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f"{REPO} exists but is not a Git checkout; choose an empty path.")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
         REPO_URL, str(REPO)], check=True,
    )
else:
    current_branch = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True,
    ).strip()
    if current_branch != BRANCH:
        raise RuntimeError(f"Expected branch {BRANCH!r}, found {current_branch!r}.")

os.chdir(REPO)
# Use the checked-out source directly as well as installing it. This keeps
# notebook imports and child CLI/test processes reliable even if Colab's
# editable-install .pth file is not reloaded into the running kernel.
PROJECT_SRC = (REPO / "src").resolve()
if not (PROJECT_SRC / "trace_rag").is_dir():
    raise FileNotFoundError(f"TRACE-RAG package source is missing: {PROJECT_SRC / 'trace_rag'}")
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))
pythonpath_entries = [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
if str(PROJECT_SRC) not in pythonpath_entries:
    os.environ["PYTHONPATH"] = os.pathsep.join([str(PROJECT_SRC), *pythonpath_entries])

GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repo:", REPO)
print("branch:", BRANCH)
print("commit:", GIT_COMMIT)

In [ ]:
# Probe CUDA in a short-lived process so the notebook kernel does not retain
# a GPU context while the project CLI launches its own process later.
probe_code = (
    "import json, torch; "
    "assert torch.cuda.is_available(), 'torch.cuda.is_available() is False'; "
    "name = torch.cuda.get_device_name(0); "
    "props = torch.cuda.get_device_properties(0); "
    "info = {'name': name, 'total_memory_gb': round(props.total_memory / (1024 ** 3), 2), "
    "'torch': torch.__version__}; "
    "print('GPU_JSON=' + json.dumps(info)); "
    "assert 't4' in name.lower(), f'Expected an NVIDIA T4, detected {name!r}'"
)
probe = subprocess.run([sys.executable, "-c", probe_code], capture_output=True, text=True)
print(probe.stdout)
if probe.returncode:
    print(probe.stderr)
    raise RuntimeError(
        "No NVIDIA T4 CUDA device was detected. Select Runtime → Change runtime type → "
        "T4 GPU, restart Colab, and rerun from the top."
    )
GPU_PROBE = probe.stdout.strip()
gpu_json_line = next(line for line in probe.stdout.splitlines() if line.startswith("GPU_JSON="))
GPU_INFO = json.loads(gpu_json_line.split("=", 1)[1])
GPU_NAME = GPU_INFO["name"]
GPU_TOTAL_GB = GPU_INFO["total_memory_gb"]
NQ_EMBED_BATCH_SIZE = 8 if GPU_TOTAL_GB >= 14 else 4
print({"cuda_gpu": GPU_NAME, "vram_gb": GPU_TOTAL_GB,
       "contriever_batch_size": NQ_EMBED_BATCH_SIZE})


In [ ]:
# Use this kernel's Python explicitly and install from the cloned checkout,
# rather than relying on whichever directory a notebook pip magic resolves.
PROJECT_INSTALL_CMD = [
    sys.executable, "-m", "pip", "install", "-q", "-e",
    f"{REPO}[data,faiss,plots,dev]", "transformers>=4.40", "sentencepiece",
]
PROJECT_INSTALL = subprocess.run(
    PROJECT_INSTALL_CMD, cwd=REPO, capture_output=True, text=True,
)
if PROJECT_INSTALL.returncode:
    print(PROJECT_INSTALL.stdout[-4000:])
    print(PROJECT_INSTALL.stderr[-4000:])
    raise RuntimeError("Project/dependency installation failed; see pip output above.")
if PROJECT_INSTALL.stdout.strip():
    print(PROJECT_INSTALL.stdout[-2000:])
if PROJECT_INSTALL.stderr.strip():
    print(PROJECT_INSTALL.stderr[-2000:])

# Editable installs add a .pth file that a running kernel may not have loaded
# yet, so reassert the checkout path before importing it in this same session.
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))
import trace_rag
print("Installed dependencies without replacing Colab PyTorch/CUDA.")
print("TRACE-RAG import:", Path(trace_rag.__file__).resolve())


## 1. Run the repository test suite

This verifies the checked-in code, including Person A/B integration, Person C attack/baseline/metric/statistics helpers, downloader query sampling, and the real-model/T4 configuration contracts. Test fixtures may be synthetic; no benchmark claim is derived from these tests. Optional hardware-dependent tests may skip when dependencies are unavailable.


In [ ]:
# For an interrupted index run in the same persistent Colab VM, set this to the
# previous run folder name (UTC timestamp) and set RESUME_INDEX=True below.
RESUME_RUN_ID = None  # example: "20261007T120000Z"
RUN_ID = RESUME_RUN_ID or datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path("runs/colab_t4") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

suite = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-ra"],
    cwd=REPO, capture_output=True, text=True,
)
TEST_LOG = RUN_DIR / "pytest_output.txt"
TEST_LOG.write_text(suite.stdout + "\n" + suite.stderr, encoding="utf-8")
print(suite.stdout)
if suite.stderr:
    print(suite.stderr)
if suite.returncode:
    raise RuntimeError(f"pytest failed; full log saved to {TEST_LOG}")
print("All non-skipped tests passed. Log:", TEST_LOG)

## 2. Start and verify a real Ollama model

Ollama is a real local LLM backend. On the T4 runtime it can offload model layers to the GPU; the diagnostic prints `ollama ps` so the actual processor placement is visible. The model is downloaded once into the runtime's Ollama cache. Ollama and CUDA embedding/NLI share the T4's VRAM, so only one Ollama model is loaded in parallel.


In [ ]:
import shutil
import urllib.request

SEED = 20261007
OLLAMA_MODEL = "qwen2.5:3b"
OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_LOG = RUN_DIR / "ollama.log"
OLLAMA_PROCESS = None

if shutil.which("ollama") is None:
    install_cmd = (
        "set -o pipefail; apt-get -qq update && apt-get -qq install -y curl zstd ca-certificates "
        "&& curl -fsSL https://ollama.com/install.sh | sh"
    )
    install = subprocess.run(["bash", "-lc", install_cmd], capture_output=True, text=True)
    if install.returncode:
        raise RuntimeError("Ollama installation failed:\n" + install.stdout[-3000:] + install.stderr[-3000:])

# Reuse a daemon if this setup cell is rerun; otherwise start one with modest
# concurrency so the T4's VRAM remains available for the CUDA pipeline.
def ollama_is_ready():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if not ollama_is_ready():
    ollama_log_handle = OLLAMA_LOG.open("a", encoding="utf-8")
    ollama_env = os.environ.copy()
    ollama_env["OLLAMA_NUM_PARALLEL"] = "1"
    ollama_env["OLLAMA_MAX_LOADED_MODELS"] = "1"
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"], stdout=ollama_log_handle, stderr=subprocess.STDOUT,
        start_new_session=True, env=ollama_env,
    )
    for _ in range(60):
        if ollama_is_ready():
            break
        time.sleep(2)
    if not ollama_is_ready():
        raise RuntimeError(f"Ollama server did not become ready; inspect {OLLAMA_LOG}")

pull = subprocess.run(["ollama", "pull", OLLAMA_MODEL], capture_output=True, text=True)
print((pull.stdout + "\n" + pull.stderr)[-4000:])
if pull.returncode:
    raise RuntimeError("Ollama model download failed.")

# Defensive local-source path in case the editable install was not picked up
# by the current Colab kernel. The setup cell also exports this for subprocesses.
PROJECT_SRC = (REPO / "src").resolve()
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))
pythonpath_entries = [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
if str(PROJECT_SRC) not in pythonpath_entries:
    os.environ["PYTHONPATH"] = os.pathsep.join([str(PROJECT_SRC), *pythonpath_entries])

from trace_rag.generation.llm import OllamaLLM
ollama_probe = OllamaLLM(OLLAMA_MODEL, base_url=OLLAMA_URL, timeout_s=180)
probe_response = ollama_probe.generate("Reply with one word: READY.", max_tokens=8)
if not probe_response.text:
    raise RuntimeError("Ollama returned an empty response; real-model check failed.")
ollama_ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
OLLAMA_PS_OUTPUT = ollama_ps.stdout.strip()
print({"ollama_model": OLLAMA_MODEL, "reply": probe_response.text,
       "latency_ms": round(probe_response.latency_ms, 1),
       "ollama_ps": OLLAMA_PS_OUTPUT})


## 3. Download the full BEIR Natural Questions corpus

The downloader keeps all corpus rows in `corpus.parquet`; it samples only query records into `queries_subset.jsonl`. It does **not** build `corpus_subset.parquet`, and it does not accumulate millions of passages in Python lists. The qrels-backed query sample is deterministic under `SEED`.

In [ ]:
NQ_DIR = Path("data/nq")
NQ_QUERY_SAMPLE_SIZE = 500
NQ_EXPECTED_PASSAGES = 2_681_468

prepare = subprocess.run(
    [sys.executable, "scripts/download_data.py", "--dataset", "nq", "--out", str(NQ_DIR),
     "--queries", str(NQ_QUERY_SAMPLE_SIZE), "--seed", str(SEED)],
    cwd=REPO, capture_output=True, text=True,
)
print(prepare.stdout[-10000:])
if prepare.returncode:
    print(prepare.stderr[-5000:])
    raise RuntimeError("Full NQ preparation failed; inspect the output above.")

import pyarrow.parquet as pq
NQ_CORPUS_ROWS = pq.ParquetFile(str(NQ_DIR / "corpus.parquet")).metadata.num_rows
NQ_QUERY_ROWS = [json.loads(line) for line in
                 (NQ_DIR / "queries_subset.jsonl").read_text(encoding="utf-8").splitlines() if line]
if NQ_CORPUS_ROWS != NQ_EXPECTED_PASSAGES:
    raise RuntimeError(
        f"Expected the full BEIR NQ corpus ({NQ_EXPECTED_PASSAGES:,} passages), "
        f"found {NQ_CORPUS_ROWS:,}. Review the dataset revision before continuing."
    )
if len(NQ_QUERY_ROWS) != NQ_QUERY_SAMPLE_SIZE:
    raise RuntimeError(f"Expected {NQ_QUERY_SAMPLE_SIZE} sampled queries, found {len(NQ_QUERY_ROWS)}")
if not (NQ_DIR / "qrels.tsv").is_file():
    raise RuntimeError("BEIR qrels.tsv is missing")
print({"corpus_file": str(NQ_DIR / "corpus.parquet"),
       "full_corpus_passages": NQ_CORPUS_ROWS,
       "sampled_real_queries": len(NQ_QUERY_ROWS),
       "qrels_file": str(NQ_DIR / "qrels.tsv")})

## 4. Full-corpus A/B/C trust stream

This runs Person A ingestion, Contriever/CUDA embeddings, FAISS retrieval, signals, and grounded answers; Person B's CUDA NLI verifier, trust ledger, policy, quarantine, and remediation; and Person C's attack transformation and stream metrics. The corpus argument is the complete 2.68M-row parquet file; only the real query stream is bounded to 500 sampled questions. Contriever uses float16 and a batch size selected from detected T4 VRAM; FAISS IVF-PQ remains CPU-resident.

The clean documents receive page-level source IDs derived from their authentic BEIR titles (`--source-from-title`) and actual import timestamps (`--live-timestamps`). Person C's attack inputs are generated perturbations of authentic gold passages, not claimed real incidents. The notebook uses calibrated pipeline settings; it does not force every document through the verifier. Ollama generation uses a real local model; check the captured `ollama ps` output to confirm whether Ollama offloaded to the T4 or fell back to CPU.


In [ ]:
NLI_MAX_LENGTH = 256
RESUME_INDEX = False  # set True only to resume indexing into this exact RUN_ID
NQ_OUT = RUN_DIR / "nq_full"
NQ_OUT.mkdir(parents=True, exist_ok=True)
os.environ["TRACE_RAG_NLI_DEVICE"] = "cuda"
os.environ["TRACE_RAG_NLI_MAX_LENGTH"] = str(NLI_MAX_LENGTH)

nq_command = [
    sys.executable, "scripts/run_trust_stream.py",
    "--config", "config/nq_t4.yaml",
    "--corpus", str(NQ_DIR / "corpus.parquet"),
    "--queries", str(NQ_DIR / "queries_subset.jsonl"),
    "--qrels", str(NQ_DIR / "qrels.tsv"),
    "--steps", "500", "--targets", "10", "--seed", str(SEED),
    "--source-from-title", "--live-timestamps",
    "--save-every", "50000", "--out", str(NQ_OUT),
    "--set", f"embedding.batch_size={NQ_EMBED_BATCH_SIZE}",
    "--set", "embedding.device=cuda",
    "--set", "embedding.dtype=float16",
    "--set", "trust.nli_mode=nli",
    "--set", "generation.backend=ollama",
    "--set", f"generation.model_name={OLLAMA_MODEL}",
    "--set", "generation.max_tokens=96",
    "--set", "generation.timeout_s=180",
]
if RESUME_INDEX:
    if not (NQ_OUT / "resolved_config.yaml").exists():
        raise RuntimeError("RESUME_INDEX=True but no prior resolved_config.yaml exists in NQ_OUT")
    nq_command.append("--skip-index")

run_env = os.environ.copy()
runner_log = NQ_OUT / "runner.log"
with runner_log.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        nq_command, cwd=REPO, env=run_env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        log_handle.write(line)
        log_handle.flush()
        if line.startswith("=") or "indexed " in line or "ingested:" in line or "checkpoint saved" in line:
            print(line, end="")
    return_code = process.wait()

if return_code:
    tail = runner_log.read_text(encoding="utf-8", errors="replace")[-12000:]
    print(tail)
    raise RuntimeError(f"Full NQ run failed; complete log is {runner_log}")

NQ_METRICS = json.loads((NQ_OUT / "metrics.json").read_text(encoding="utf-8"))
assert NQ_METRICS["generation_backend"] == "ollama", NQ_METRICS
assert NQ_METRICS["generation_model"] == OLLAMA_MODEL, NQ_METRICS
assert NQ_METRICS["verifier_mode"] == "nli", NQ_METRICS
assert str(NQ_METRICS["embedding_device"]).startswith("cuda"), NQ_METRICS
assert str(NQ_METRICS["verifier_device"]).startswith("cuda"), NQ_METRICS
assert NQ_METRICS["clean_source_mode"] == "title_based", NQ_METRICS
assert NQ_METRICS["live_timestamps"] is True, NQ_METRICS
RUN_METRICS = [{"dataset": "BEIR NQ full corpus / 500-query stream", "metrics": NQ_METRICS}]
print("Resolved execution:", json.dumps({k: NQ_METRICS.get(k) for k in (
    "embedding_model", "embedding_device", "verifier_mode", "verifier_device",
    "generation_backend", "generation_model", "steps", "poison_documents",
    "poison_quarantined", "clean_false_quarantine", "mean_latency_ms",
)}, indent=2))
# Free the Ollama model's T4 allocation before the separate Person C baseline
# diagnostic loads retrieval/NLI models in the notebook kernel.
ollama_stop = subprocess.run(["ollama", "stop", OLLAMA_MODEL], capture_output=True, text=True)
OLLAMA_STOP_OUTPUT = (ollama_stop.stdout + "\n" + ollama_stop.stderr).strip()
print({"ollama_model_unload_returncode": ollama_stop.returncode,
       "ollama_model_unload": OLLAMA_STOP_OUTPUT})


## 5. Person C baseline API diagnostic on a real NQ query

The cell below invokes **every implemented Person C baseline**—no defence, perplexity, duplicate filtering, TrustRAG, RobustRAG, and always-on leave-one-out—on the same authentic NQ retrieval result. The perplexity scorer is a real DistilGPT-2 model. It also reports BEIR qrels retrieval metrics.

This is deliberately a one-query API/integration diagnostic, not a valid head-to-head attack benchmark: the perplexity cutoff is calibrated on that query's retrieved pool, the baseline policies are not run over matched attack/seed matrices, and the result must not be presented as comparative performance. The repository unit suite covers the broader attack/evaluation/statistics APIs.

In [ ]:
import tempfile
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from trace_rag.baselines import (
    BASELINE_NAMES, BaselineResult, RobustRAGPolicy, TrustRAGPolicy, always_on_loo,
    duplicate_filter, no_defence, perplexity_filter, trust_threshold_filter,
)
from trace_rag.config import Config
from trace_rag.evaluation import retrieval_metrics
from trace_rag.pipeline import PersonAPipeline
from trace_rag.trust.ledger import TrustLedger

baseline_config = Config.load(NQ_OUT / "resolved_config.yaml")
# Isolate this diagnostic from the trust state produced by the stream run while
# reusing its authentic full-corpus provenance store and FAISS index.
baseline_temp = tempfile.TemporaryDirectory(prefix="trace-rag-baseline-")
baseline_config.storage.trust_db = str(Path(baseline_temp.name) / "trust.sqlite3")
baseline_pipeline = PersonAPipeline.from_config(baseline_config, load_existing_index=True)

try:
    query_row = NQ_QUERY_ROWS[0]
    query_id = str(query_row["qid"])
    query_text = str(query_row["text"])
    if baseline_pipeline.verifier is None:
        raise RuntimeError("The real NLI verifier was not wired into the baseline diagnostic")
    real_result = baseline_pipeline.answer(query_text, query_id=query_id, log=False, now=time.time())
    real_docs = list(real_result.retrieval.documents)
    if not real_docs:
        raise RuntimeError("The NQ baseline diagnostic query returned no passages")

    ppl_tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
    ppl_model = AutoModelForCausalLM.from_pretrained("distilgpt2").to("cpu").eval()

    def real_perplexity(text):
        tokens = ppl_tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
        if tokens["input_ids"].shape[1] < 2:
            return 1.0
        with torch.inference_mode():
            loss = ppl_model(**tokens, labels=tokens["input_ids"]).loss
        return float(torch.exp(loss).item())

    real_ppl_scores = [real_perplexity(document.text) for document in real_docs]
    # This within-query threshold only checks the baseline API; it is not a
    # calibrated research threshold or an independent validation split.
    ppl_threshold = float(np.quantile(real_ppl_scores, 0.95))
    baseline_outputs = {
        "no_defence": no_defence(real_docs),
        "duplicate_filter": duplicate_filter(real_docs),
        "perplexity": perplexity_filter(real_docs, real_perplexity, ppl_threshold),
        "always_on_loo": always_on_loo(
            query_text, query_id, real_docs, baseline_pipeline.verifier,
        ),
        "trust_threshold_filter": trust_threshold_filter(real_docs, threshold=0.20),
    }
    relevant_ids = set()
    with (NQ_DIR / "qrels.tsv").open(encoding="utf-8") as qrels_handle:
        for line in qrels_handle:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3 and parts[0] == query_id:
                try:
                    if float(parts[2]) > 0:
                        relevant_ids.add(parts[1])
                except ValueError:
                    pass

    def metric_payload(documents):
        retrieved_ids = [doc.doc_id.split("#", 1)[0] for doc in documents]
        return retrieval_metrics(retrieved_ids, relevant_ids).to_dict()

    for policy_name, policy_type in (("TrustRAG", TrustRAGPolicy), ("RobustRAG", RobustRAGPolicy)):
        isolated_ledger = TrustLedger(
            Path(baseline_temp.name) / f"{policy_name.lower()}_trust.sqlite3",
            store=baseline_pipeline.store,
            config=baseline_config.trust.to_ledger_config(),
        )
        try:
            if policy_name == "TrustRAG":
                policy = policy_type(ledger=isolated_ledger)
            else:
                policy = policy_type(ledger=isolated_ledger, trust_floor=0.20)
            decision = policy.decide(
                query_text, query_id, real_docs, real_result.assessments,
                verifier=baseline_pipeline.verifier,
                pool=real_result.retrieval.pool, now=time.time(),
            )
            keep_ids = set(decision.context_doc_ids)
            kept = [doc for doc in real_docs if doc.doc_id in keep_ids]
            baseline_outputs[policy_name] = BaselineResult(
                documents=kept,
                blocked=list(decision.excluded_doc_ids),
                verified=list(decision.verified),
                llm_calls=sum(item.llm_calls for item in decision.verified),
            )
        finally:
            isolated_ledger.close()

    if not BASELINE_NAMES or not set(BASELINE_NAMES).issubset(baseline_outputs):
        raise AssertionError(f"Not all baseline policies ran: {sorted(baseline_outputs)}")
    baseline_payload = {
        "scope": "one-query real-NQ baseline API smoke; not a benchmark matrix",
        "corpus_passages_indexed": NQ_CORPUS_ROWS,
        "query_id": query_id,
        "query": query_text,
        "relevant_passages_in_qrels": len(relevant_ids),
        "perplexity_model": "distilgpt2",
        "perplexity_threshold": ppl_threshold,
        "perplexity_threshold_note": "95th percentile of the same query's retrieved docs; diagnostic only",
        "baselines": {},
    }
    for name, outcome in baseline_outputs.items():
        documents = outcome.documents
        baseline_payload["baselines"][name] = {
            "retrieved_doc_ids": [doc.doc_id for doc in documents],
            "blocked_doc_ids": list(outcome.blocked),
            "retrieval_metrics": metric_payload(documents),
            "llm_calls": int(outcome.llm_calls),
        }
    BASELINE_PATH = RUN_DIR / "baseline_smoke.json"
    BASELINE_PATH.write_text(json.dumps(baseline_payload, indent=2), encoding="utf-8")
    print(json.dumps(baseline_payload, indent=2))
finally:
    baseline_pipeline.close()
    baseline_temp.cleanup()
    try:
        del ppl_model, ppl_tokenizer
    except NameError:
        pass
    gc.collect()

## 6. Save reproducibility metadata and artifacts

The archive includes the test log, full-run resolved config/metrics/stream/trust history/figures, Ollama startup log, `ollama ps` placement output, and one-query baseline diagnostic. It does not include the multi-gigabyte NQ source corpus or model caches.


In [ ]:
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"

summary_fields = [
    "dataset", "seed", "embedding_model", "embedding_device", "verifier_mode",
    "verifier_device", "generation_backend", "generation_model", "steps",
    "poison_retrieved_rate", "poison_citation_rate", "poison_quarantined",
    "clean_false_quarantine", "mean_latency_ms", "mean_llm_calls_per_query",
]
summary_rows = []
for run in RUN_METRICS:
    row = {"dataset": run["dataset"]}
    row.update({key: run["metrics"].get(key) for key in summary_fields if key != "dataset"})
    summary_rows.append(row)

summary_path = RUN_DIR / "presentation_summary.csv"
with summary_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=summary_fields)
    writer.writeheader()
    writer.writerows(summary_rows)

metadata = {
    "title": "TRACE-RAG full BEIR NQ Colab T4/Ollama integration run",
    "run_id_utc": RUN_ID,
    "git_branch": BRANCH,
    "git_commit": GIT_COMMIT,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": package_version("torch"),
    "transformers": package_version("transformers"),
    "pyarrow": package_version("pyarrow"),
    "faiss_cpu": package_version("faiss-cpu"),
    "gpu_probe": GPU_PROBE,
    "gpu_info": GPU_INFO,
    "contriever_dtype": "float16",
    "contriever_batch_size": NQ_EMBED_BATCH_SIZE,
    "ollama_ps": OLLAMA_PS_OUTPUT,
    "ollama_unload": OLLAMA_STOP_OUTPUT,
    "seed": SEED,
    "corpus": "BEIR Natural Questions",
    "corpus_passages": NQ_CORPUS_ROWS,
    "sampled_qrels_queries": len(NQ_QUERY_ROWS),
    "embedding_model": "facebook/contriever",
    "answer_backend": "ollama",
    "answer_model": OLLAMA_MODEL,
    "answer_accelerator_note": "Ollama placement is recorded in ollama_ps; embeddings and NLI must report CUDA in metrics",
    "clean_provenance_note": (
        "BEIR NQ lacks authorship and original event timestamps; clean source IDs derive from "
        "authentic page titles and import times use wall-clock time. Controlled poison "
        "payloads and attacker identity are generated from authentic qrels gold passages."
    ),
    "scope_warning": (
        "Full passage corpus, sampled query stream, and one-query baseline API diagnostic; "
        "not a complete attack x baseline x seed benchmark or production certification."
    ),
    "baseline_diagnostic": str(BASELINE_PATH),
    "runs": [{"dataset": item["dataset"], "metrics": item["metrics"]}
             for item in RUN_METRICS],
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(RUN_DIR / "RESULTS_README.md").write_text(
    "# TRACE-RAG full-corpus integration artifacts\n\n"
    "Authentic BEIR NQ passages were indexed; query traffic was sampled. Generated poison "
    "mutations are controlled experiments based on qrels-relevant real passages, not real "
    "incidents. BEIR NQ lacks authorship and original timestamps; clean source IDs derive from real page titles. The Person C "
    "baseline output is a one-query API diagnostic, not a matched benchmark matrix. See "
    "`run_metadata.json` for devices, model IDs, code revision, and limitations.\n",
    encoding="utf-8",
)

archive = RUN_DIR.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
    for artifact in sorted(RUN_DIR.rglob("*")):
        if artifact.is_file():
            zipped.write(artifact, artifact.relative_to(RUN_DIR.parent))

print("Summary CSV:", summary_path)
print("Metadata:", RUN_DIR / "run_metadata.json")
print("Baseline diagnostic:", BASELINE_PATH)
print("Results archive:", archive)
print(summary_path.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(archive)))
print("Download the ZIP, or copy it into Google Drive using the next cell.")

In [ ]:
# Optional persistence to Google Drive. Set SAVE_TO_DRIVE=True before running.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/TRACE-RAG") / archive.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, destination)
    print("Copied results to:", destination)

## Reading and presenting the output

- Record the exact commit, full-corpus passage count, sampled-query count, seed, Contriever/NLI/Ollama model IDs, detected T4 name/VRAM, resolved CUDA devices, selected embedding batch size, and `ollama ps` processor placement from `run_metadata.json` and `nq_full/metrics.json`.
- Label results as full-corpus integration / sampled-query smoke. The notebook does **not** run every query, every attack family, each baseline across the same poisoned stream, or multiple independent seeds.
- The generated attacker identity and modified gold passages are controlled test cases, not organic incidents. The one-query baseline file checks that implementations run on authentic NQ retrievals; it is not comparative evidence.
- BEIR NQ provides no original contributor provenance or timestamps. The notebook uses authentic page titles rather than fabricated contributor identities and wall-clock import times rather than fake historical ages, but controlled attack events are still generated. For a strict no-generated-data study, supply real source metadata and real poisoning incidents.
- T4 selection alone is not proof of CUDA use; verify both `embedding_device` and `verifier_device` are CUDA in the metrics. Check `ollama ps` separately because Ollama can fall back to CPU.
- TRACE-RAG is a research prototype, **not production-ready for company deployment**. See `docs/PRODUCTION_READINESS.md` and `docs/PERSON_C_TODO.md`.
